## ROS + Interface Setup

In [1]:
import rospy 
import actionlib
from assignment_2_2024.msg import PlanningAction, PlanningGoal
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan
from Simulation_pkg.msg import PositionVelocity
import threading
import time
import ipywidgets as widgets
from ipywidgets import interactive
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, HTML

# ROS Node
rospy.init_node('jupyter_action_client', anonymous=True, disable_signals=True)

# Globals
position_velocity_pub = rospy.Publisher("position_velocity", PositionVelocity, queue_size=10)
pos_vel_msg = PositionVelocity()
ac = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
ac_lock = threading.Lock()
goal_reached = False
request_coordinates = False
reached_count = 0
cancel_count = 0
goal_sent_flag = False



## Interface Widgets

In [2]:
status_output = widgets.Output()
position_display = widgets.HTML("<b>Current Position:</b> Waiting for data...")
velocity_display = widgets.HTML("<b>Current Velocity:</b> Waiting for data...")
obstacle_distance_display = widgets.HTML("<b>Closest Obstacle:</b> Waiting for data...")
target_display = widgets.HTML("<b>Target Position:</b> None")

x_input = widgets.FloatText(value=0.0, description='X:')
y_input = widgets.FloatText(value=0.0, description='Y:')

set_goal_button = widgets.Button(description='Set Goal', button_style='success', icon='check')
cancel_button = widgets.Button(description='Cancel Goal', button_style='danger', icon='stop')

## ROS Callbacks

In [3]:
def odom_callback(msg):
    pos_vel_msg.x = msg.pose.pose.position.x
    pos_vel_msg.y = msg.pose.pose.position.y
    pos_vel_msg.vel_x = msg.twist.twist.linear.x
    pos_vel_msg.vel_z = msg.twist.twist.angular.z

    position_display.value = f"<b>Current Position:</b> X: {pos_vel_msg.x:.2f}, Y: {pos_vel_msg.y:.2f}"
    velocity_display.value = f"<b>Current Velocity:</b> Linear: {pos_vel_msg.vel_x:.2f}, Angular: {pos_vel_msg.vel_z:.2f}"
    position_velocity_pub.publish(pos_vel_msg)
    #x_path.append(pos_vel_msg.x)
    #y_path.append(pos_vel_msg.y)

def scan_callback(msg):
    valid_ranges = [r for r in msg.ranges if not rospy.is_shutdown() and not (r == float('inf') or r != r)]
    if valid_ranges:
        closest = min(valid_ranges)
        obstacle_distance_display.value = f"<b>Closest Obstacle:</b> {closest:.2f} meters"
    else:
        obstacle_distance_display.value = "<b>Closest Obstacle:</b> No valid data"

## Goal Monitor Thread

In [4]:
x_path = []
y_path = []

def monitor_goal_status():
    global goal_reached, request_coordinates, reached_count, aborted_count, goal_sent_flag
    rate = rospy.Rate(10)
    while not rospy.is_shutdown():
        with ac_lock:
            if ac:
                state = ac.get_state()
                if state == actionlib.GoalStatus.SUCCEEDED and not goal_reached:
                    reached_count += 1
                    with status_output:
                        clear_output(wait=True)
                        print("Target reached!")
                    goal_reached = True
                    request_coordinates = False
                    goal_sent_flag = False
                elif state == actionlib.GoalStatus.ABORTED:
                    with status_output:
                        clear_output(wait=True)
                        print("Target could not be reached. Please set a new goal.")
                    goal_reached = True
                    request_coordinates = False
        rate.sleep()

## Goal Control Buttons

In [5]:
def on_set_goal_button_clicked(b):
    global goal_reached
    global goal_sent_flag
    
    x = x_input.value
    y = y_input.value

    if not (isinstance(x, float) and isinstance(y, float) and x == x and y == y):
        with status_output:
            clear_output(wait=True)
            print("Invalid input. Both x and y must be finite numbers.")
        return

    pos_vel_msg.target_x = x
    pos_vel_msg.target_y = y
    goal = PlanningGoal()
    goal.target_pose.header.frame_id = "map"
    goal.target_pose.header.stamp = rospy.Time.now()
    goal.target_pose.pose.position.x = x
    goal.target_pose.pose.position.y = y
    goal.target_pose.pose.orientation.w = 1.0

    with ac_lock:
        ac.send_goal(goal)

    target_display.value = f"<b>Target Position:</b> X: {x:.2f}, Y: {y:.2f}"
    goal_reached = False
    with status_output:
        clear_output(wait=True)
        print(f"Goal sent to ({x:.2f}, {y:.2f}).")
        goal_sent_flag = True

def on_cancel_button_clicked(b):
    global request_coordinates, goal_sent_flag, cancel_count
    
    if goal_sent_flag:
        with ac_lock:
            ac.cancel_goal()
        with status_output:
            clear_output(wait=True)
            print("Goal canceled.")
        request_coordinates = False


        cancel_count += 1

        goal_sent_flag = False




## Display UI

In [6]:
# Styled info displays
position_display = widgets.HTML(value="<b>Current Position:</b> X: ---, Y: ---", layout=widgets.Layout(padding='5px'))
velocity_display = widgets.HTML(value="<b>Current Velocity:</b> Linear: ---, Angular: ---", layout=widgets.Layout(padding='5px'))
obstacle_distance_display = widgets.HTML(value="<b>Closest Obstacle:</b> ---", layout=widgets.Layout(padding='5px'))
target_display = widgets.HTML(value="<b>Target Position:</b> X: ---, Y: ---", layout=widgets.Layout(padding='5px'))

status_output = widgets.Output(value="<b>Status:</b> Waiting for action server...", layout=widgets.Layout(padding='10px', border='1px solid gray'))

# Section titles
style_header = lambda text: widgets.HTML(value=f"<h4 style='color:#2E86C1; margin: 10px 0;'>{text}</h3>")

# Create sliders
x_slider = widgets.FloatSlider(
    value=0.0, min=-10.0, max=10.0, step=0.1,
    description='X Target:', style={'description_width': '100px'},
    continuous_update=False, layout=widgets.Layout(width='90%')
)

y_slider = widgets.FloatSlider(
    value=0.0, min=-10.0, max=10.0, step=0.1,
    description='Y Target:', style={'description_width': '100px'},
    continuous_update=False, layout=widgets.Layout(width='90%')
)

# Create input fields
x_input = widgets.FloatText(value=0.0, description="X Input:", style={'description_width': '100px'})
y_input = widgets.FloatText(value=0.0, description="Y Input:", style={'description_width': '100px'})

# Sync sliders and inputs
def update_inputs_from_sliders(x, y):
    x_input.value = x
    y_input.value = y

def update_sliders_from_inputs(change):
    x_slider.value = x_input.value
    y_slider.value = y_input.value

x_slider.observe(lambda change: update_inputs_from_sliders(x_slider.value, y_slider.value), names='value')
y_slider.observe(lambda change: update_inputs_from_sliders(x_slider.value, y_slider.value), names='value')
x_input.observe(update_sliders_from_inputs, names='value')
y_input.observe(update_sliders_from_inputs, names='value')

# Input and slider layout
input_controls = widgets.VBox([
    style_header("🎯 Set Target"),
    widgets.HBox([x_slider, x_input]),
    widgets.HBox([y_slider, y_input]),
], layout=widgets.Layout(padding='10px', border='1px solid lightgray', border_radius='10px', flex='1 1 auto', height='auto'))

# Buttons
set_goal_button = widgets.Button(description="🚀 Set Goal", button_style='success', layout=widgets.Layout(width='150px'))
set_goal_button.on_click(on_set_goal_button_clicked)
cancel_button = widgets.Button(description="✖ Cancel", button_style='danger', layout=widgets.Layout(width='150px'))
cancel_button.on_click(on_cancel_button_clicked)
button_controls = widgets.HBox([set_goal_button, cancel_button], layout=widgets.Layout(justify_content='center', padding='10px'))

# Info display block
info_widgets = widgets.VBox([
    style_header("📡 Robot Info"),
    position_display,
    velocity_display,
    obstacle_distance_display,
    target_display
], layout=widgets.Layout(padding='10px', border='1px solid lightgray', border_radius='10px', flex='1 1 auto', height='auto'))


# Final layout with updated height and overflow control
ui = widgets.VBox([
    info_widgets,
    input_controls,
    button_controls,
    status_output
], layout=widgets.Layout(
    padding='20px',
    border='2px solid #D6EAF8',
    border_radius='10px',
    background_color='#FBFCFC',
    width='100%',           
    height='auto',              
    overflow='auto'            
))

# ----- Connection logic to Action Server -----
try:
    status_output.value = "<b>Status:</b> Connecting to action server..."

    ac.wait_for_server()  # Assumes 'ac' is your SimpleActionClient
    status_output.value = "<b>Status:</b> ✅ Action server available."

    # Start monitoring thread
    monitor_thread = threading.Thread(target=monitor_goal_status)
    monitor_thread.daemon = True
    monitor_thread.start()

    # Subscribe to odometry and scan
    odom_sub = rospy.Subscriber("/odom", Odometry, odom_callback)
    scan_sub = rospy.Subscriber("/scan", LaserScan, scan_callback)

    # Display the UI
    display(ui)

except rospy.ROSInterruptException:
    print("ROS node interrupted")
    status_output.value = "<b>Status:</b> ❌ ROS node interrupted"
except Exception as e:
    print(f"Error: {str(e)}")
    status_output.value = f"<b>Status:</b> ❌ Error: {str(e)}"

## Target counter plot

In [11]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib as mpl

# Apply light clean styling manually
mpl.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.edgecolor': '#CCCCCC',
    'axes.linewidth': 1.0,
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'figure.facecolor': 'white',
    'axes.facecolor': '#FAFAFA',
})

output = widgets.Output(layout=widgets.Layout(
    display='flex',
    justify_content='center',
    align_items='center',
    border='solid 1px #E0E0E0',
    padding='20px',
    margin='10px',
    height='auto',  # <-- Let it grow as needed
    flex='1 1 auto'
))

def plot_reach_status():
    with output:
        clear_output(wait=True)
        labels = ['Reached', 'Canceled']
        values = [reached_count, cancel_count]
        colors = ['#4CAF50', '#F44336']

        fig, ax = plt.subplots(figsize=(6, 4))
        bars = ax.bar(labels, values, color=colors, edgecolor='black')
        ax.set_title("Target Reach Status", fontsize=16, pad=15, fontname="DejaVu Sans")
        ax.set_ylim(0, ((max(reached_count, cancel_count) // 10) + 1) * 10)
        
        for bar in bars:
            yval = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, int(yval),
                    ha='center', va='bottom', fontweight='bold', color='#333333')
        
        plt.tight_layout()  # <-- Ensures the layout fits cleanly
        plt.show()

refresh_button = widgets.Button(
    description="🔄 Refresh",
    button_style='info',
    layout=widgets.Layout(width='150px', height='45px'),
    style={'font_weight': 'bold'}
)

reset_button = widgets.Button(
    description="♻️ Reset Counts",
    button_style='danger',
    layout=widgets.Layout(width='150px', height='45px'),
    style={'font_weight': 'bold'}
)

def on_refresh_button_clicked(b):
    plot_reach_status()

def on_reset_button_clicked(b):
    global reached_count, cancel_count
    reached_count = 0
    cancel_count = 0
    plot_reach_status()

refresh_button.on_click(on_refresh_button_clicked)
reset_button.on_click(on_reset_button_clicked)

button_box = widgets.HBox(
    [reset_button, refresh_button],
    layout=widgets.Layout(
        justify_content='center',
        gap='20px',
        padding='10px'
    )
)

display(widgets.VBox([
    widgets.HTML("<h3 style='text-align:center;'>📊 Goal Status Dashboard</h3>"),
    output,
    button_box
], layout=widgets.Layout(
    align_items='center',
    padding='20px',
    background_color='#F5F5F5',
    border_radius='10px',
    box_shadow='0px 0px 10px rgba(0,0,0,0.1)'
)))

refresh_button.click()



## Robot path live

In [8]:
%matplotlib widget

import matplotlib.pyplot as plt
import rospy
from nav_msgs.msg import Odometry
import numpy as np
from matplotlib.animation import FuncAnimation

class Visualiser:
    def __init__(self):
        self.fig, self.ax = plt.subplots()
        self.ln, = self.ax.plot([], [], 'ro', markersize=6)  # Red markers for the robot
        self.x_data, self.y_data = [], []

        # Set plot background and other aesthetics
        self.ax.set_facecolor('#2d3b4f')  # Dark background
        self.fig.patch.set_facecolor('#2d3b4f')  # Dark background
        self.ax.set_xlim(-10, 10)
        self.ax.set_ylim(-10, 10)
        self.ax.grid(True, color='gray', linestyle='--', linewidth=0.5)  # Add grid
        self.ax.set_xlabel("X Position", fontsize=12, color='white')  # X-axis label
        self.ax.set_ylabel("Y Position", fontsize=12, color='white')  # Y-axis label
        self.ax.set_title("Robot Position Over Time", fontsize=14, color='white')  # Title
        self.ax.tick_params(axis='both', labelsize=10, labelcolor='white')  # Axis ticks

        legend = self.ax.legend(
            ['Robot Path'],
            loc='lower left',
            bbox_to_anchor=(-0.18, 1.08),
            fontsize=8,
            frameon=False
        )
        for text in legend.get_texts():
            text.set_color('white')



    def plot_init(self):
        """This will run once to initialize the plot."""
        self.ln.set_data([], [])
        return self.ln,

    def odom_callback(self, msg):
        """This is the callback for the Odometry message."""
        self.y_data.append(msg.pose.pose.position.y)
        self.x_data.append(msg.pose.pose.position.x)

    def update_plot(self, frame):
        """This updates the plot with new data."""
        self.ln.set_data(self.x_data, self.y_data)
        return self.ln,  # Update the plot with the new data


# Setup the ROS node
# rospy.init_node('odom_visualizer_node')  # Uncomment when running on ROS
vis = Visualiser()

# Subscribe to Odometry data
sub = rospy.Subscriber('/odom', Odometry, vis.odom_callback)

# Create the animation
ani = FuncAnimation(vis.fig, vis.update_plot, init_func=vis.plot_init, blit=True)

plt.show(block=True)


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Cleanup

In [9]:
# === [9] Cleanup ===
def cleanup():
    try:
        rospy.signal_shutdown("Notebook closing")
        print("ROS node shutdown")
    except:
        pass

import atexit
atexit.register(cleanup)

<function __main__.cleanup()>